# Notebook 03: Descriptive Analysis

## Purpose

Describe the analytic sample, compare included and excluded adults, examine target concordance and discordant groups, inspect subgroup cell sizes, and generate descriptive tables and diagnostic figures.

## Inputs

- Adult analysis base and labelled complete-case data from Notebooks 01--02
- Sample and target metadata from Notebooks 01--02

## Outputs

- Paper-ready analytic-sample and target-concordance source tables
- Included-versus-excluded and discordant-group summaries
- Diagnostic distribution and boxplot figures
- `data/processed/descriptive_analysis_metadata.json`

## Dependencies

Run Notebooks 01 and 02 first. Notebook 09 uses selected descriptive outputs.

> **Repository policy:** Notebook outputs and execution counts are cleared in the public source files. Run the notebooks in the documented order to regenerate all results.

## 1. Setup

In [ ]:
from pathlib import Path
import json
import math

import numpy as np
import pandas as pd

try:
    import matplotlib.pyplot as plt
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError(
        "This notebook requires matplotlib. Install it in the active "
        "environment with: python -m pip install matplotlib"
    ) from exc

from IPython.display import display

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 250)
pd.set_option("display.width", 160)


## 2. Define project paths

In [ ]:
PROJECT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
OUTPUT_DIR = PROJECT_DIR / "outputs"
TABLE_DIR = OUTPUT_DIR / "tables"
FIGURE_DIR = OUTPUT_DIR / "figures"

TABLE_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

analysis_base_path = PROCESSED_DIR / "nhanes_diabetes_analysis_base.csv"
labelled_complete_case_path = (
    PROCESSED_DIR / "nhanes_diabetes_complete_case_labeled.csv"
)
sample_metadata_path = PROCESSED_DIR / "sample_metadata.json"

target_prevalence_path = TABLE_DIR / "target_prevalence.csv"
label_group_counts_path = TABLE_DIR / "label_group_counts.csv"

required_input_paths = [
    analysis_base_path,
    labelled_complete_case_path,
    sample_metadata_path,
    target_prevalence_path,
    label_group_counts_path,
]

missing_input_paths = [
    path for path in required_input_paths if not path.exists()
]

if missing_input_paths:
    missing_text = "\n".join(f"- {path}" for path in missing_input_paths)
    raise FileNotFoundError(
        "Required outputs from Notebooks 01 and 02 are missing:\n"
        f"{missing_text}\n"
        "Run 01_download_and_merge.ipynb and "
        "02_sample_and_labels.ipynb before this notebook."
    )

print("Project directory:", PROJECT_DIR)
print("Processed data directory:", PROCESSED_DIR)
print("Table output directory:", TABLE_DIR)
print("Figure output directory:", FIGURE_DIR)


## 3. Load the processed data and upstream results

In [ ]:
analysis_base = pd.read_csv(analysis_base_path)
complete_case = pd.read_csv(labelled_complete_case_path)

with sample_metadata_path.open("r", encoding="utf-8") as file:
    sample_metadata = json.load(file)

target_prevalence_from_notebook02 = pd.read_csv(target_prevalence_path)
label_group_counts_from_notebook02 = pd.read_csv(
    label_group_counts_path,
    index_col=0,
)

print("Adult analysis-base shape:", analysis_base.shape)
print("Labelled complete-case shape:", complete_case.shape)
print("Notebook 01 metadata:")
print(sample_metadata)


## 4. Validate the loaded datasets

This checkpoint verifies that Notebook 03 uses exactly the adult analysis base and labelled complete-case sample created upstream.

In [ ]:
required_analysis_base_columns = {
    "id",
    "age",
    "sex",
    "race_ethnicity",
    "pregnancy_status",
    "confirmed_current_pregnancy",
    "primary_sample_eligible",
    "income_poverty_ratio",
    "bmi",
    "currently_insured",
    "confirmed_current_pregnancy",
    "primary_sample_eligible",
    "insurance_history",
    "hba1c",
    "self_reported_prior_diagnosis",
    "current_hba1c_ge_6_5",
    "blood_test_past_3y",
    "insulin_now",
    "diabetes_pills_now",
    "any_diabetes_medication",
    "exam_status",
    "phlebotomy_weight",
    "survey_stratum",
    "survey_psu",
}

required_complete_case_columns = required_analysis_base_columns.union(
    {"label_group", "joint_label_code"}
)

missing_base_columns = required_analysis_base_columns.difference(
    analysis_base.columns
)
missing_complete_columns = required_complete_case_columns.difference(
    complete_case.columns
)

if missing_base_columns:
    raise KeyError(
        "The adult analysis base is missing columns: "
        f"{sorted(missing_base_columns)}"
    )

if missing_complete_columns:
    raise KeyError(
        "The labelled complete-case dataset is missing columns: "
        f"{sorted(missing_complete_columns)}"
    )

if not analysis_base["id"].is_unique:
    raise ValueError("The adult analysis base contains duplicate IDs.")

if not complete_case["id"].is_unique:
    raise ValueError("The complete-case dataset contains duplicate IDs.")

if not set(complete_case["id"]).issubset(set(analysis_base["id"])):
    raise ValueError(
        "Complete-case participant IDs are not a subset of the adult base."
    )

if len(analysis_base) != int(sample_metadata["n_adults"]):
    raise ValueError(
        "Adult analysis-base size does not match Notebook 01 metadata."
    )

if len(complete_case) != int(sample_metadata["n_complete_case"]):
    raise ValueError(
        "Complete-case size does not match Notebook 01 metadata."
    )

# Notebook 01 must convert SAS/XPT numeric missing-value placeholders to NaN.
# Values such as 5.397605e-79 are not genuine near-zero measurements.
placeholder_check_columns = [
    "income_poverty_ratio",
    "bmi",
    "hba1c",
    "phlebotomy_weight",
]

for dataset_name, dataframe in {
    "adult analysis base": analysis_base,
    "labelled complete-case data": complete_case,
}.items():
    for column in placeholder_check_columns:
        placeholder_mask = (
            dataframe[column].notna()
            & dataframe[column].gt(0)
            & dataframe[column].lt(1e-50)
        )

        if placeholder_mask.any():
            raise ValueError(
                f"The {dataset_name} contains an SAS/XPT numeric "
                f"missing-value placeholder in {column}. Rerun the corrected "
                "Notebooks 01 and 02 before running Notebook 03."
            )

integer_like_columns = [
    "id",
    "exam_status",
    "survey_stratum",
    "survey_psu",
    "self_reported_prior_diagnosis",
    "current_hba1c_ge_6_5",
    "currently_insured",
    "blood_test_past_3y",
    "insulin_now",
    "diabetes_pills_now",
    "any_diabetes_medication",
]

for dataframe in [analysis_base, complete_case]:
    for column in integer_like_columns:
        dataframe[column] = pd.to_numeric(
            dataframe[column],
            errors="coerce",
        ).astype("Int64")

string_columns = [
    "sex",
    "race_ethnicity",
    "insurance_history",
    "pregnancy_status",
]

for dataframe in [analysis_base, complete_case]:
    for column in string_columns:
        dataframe[column] = dataframe[column].astype("string")

target_columns = [
    "self_reported_prior_diagnosis",
    "current_hba1c_ge_6_5",
]

predictor_columns = [
    "age",
    "sex",
    "race_ethnicity",
    "income_poverty_ratio",
    "bmi",
    "insurance_history",
]

survey_columns = [
    "exam_status",
    "phlebotomy_weight",
    "survey_stratum",
    "survey_psu",
]

required_complete_values = (
    predictor_columns + target_columns + survey_columns
)

if complete_case[required_complete_values].isna().sum().sum() != 0:
    raise ValueError(
        "The labelled complete-case dataset contains missing required values."
    )

for target in target_columns:
    target_values = set(complete_case[target].dropna().astype(int).unique())
    if not target_values.issubset({0, 1}):
        raise ValueError(
            f"{target} contains values other than 0 and 1: {target_values}"
        )


if complete_case["confirmed_current_pregnancy"].ne(0).any():
    raise ValueError(
        "The labelled complete-case dataset contains a confirmed current "
        "pregnancy."
    )

if not complete_case["primary_sample_eligible"].eq(1).all():
    raise ValueError(
        "The labelled complete-case dataset contains a participant who is "
        "not eligible for the primary analysis."
    )

if int(analysis_base["confirmed_current_pregnancy"].sum()) != int(
    sample_metadata["n_confirmed_current_pregnancy_excluded"]
):
    raise ValueError(
        "The confirmed-pregnancy count does not match Notebook 01 metadata."
    )

if not complete_case["exam_status"].eq(2).all():
    raise ValueError(
        "At least one complete-case participant did not complete the MEC exam."
    )

if not complete_case["phlebotomy_weight"].gt(0).all():
    raise ValueError(
        "At least one complete-case participant has a non-positive phlebotomy weight."
    )

expected_joint_code = (
    "D"
    + complete_case["self_reported_prior_diagnosis"].astype(str)
    + "_H"
    + complete_case["current_hba1c_ge_6_5"].astype(str)
)

if not expected_joint_code.eq(
    complete_case["joint_label_code"].astype(str)
).all():
    raise ValueError(
        "At least one joint-label code disagrees with the two target columns."
    )

print("Validation passed.")


## 5. Analysis labels and fixed category orders

In [ ]:
TARGET_DISPLAY_NAMES = {
    "self_reported_prior_diagnosis": "Prior reported clinician diagnosis",
    "current_hba1c_ge_6_5": "Current HbA1c at least 6.5%",
}

JOINT_LABEL_ORDER = [
    "D0_H0",
    "D0_H1",
    "D1_H0",
    "D1_H1",
]

JOINT_LABEL_SHORT = {
    "D0_H0": "No diagnosis / HbA1c < 6.5",
    "D0_H1": "No diagnosis / HbA1c ≥ 6.5",
    "D1_H0": "Diagnosis / HbA1c < 6.5",
    "D1_H1": "Diagnosis / HbA1c ≥ 6.5",
}

JOINT_LABEL_INTERPRETATION = {
    "D0_H0": (
        "Neither operational label indicates diabetes. This does not prove "
        "the absence of diabetes under every possible clinical definition."
    ),
    "D0_H1": (
        "Current HbA1c is elevated without a reported prior diagnosis. This "
        "may indicate potentially unrecognised elevated glycaemia, but one "
        "measurement is not definitive proof of undiagnosed diabetes."
    ),
    "D1_H0": (
        "A prior diagnosis is reported while current HbA1c is below 6.5%. "
        "This may reflect treatment or current glycaemic control and should "
        "not automatically be interpreted as diagnostic error."
    ),
    "D1_H1": (
        "Both operational labels are positive: a prior diagnosis is reported "
        "and current HbA1c is at least 6.5%."
    ),
}

CATEGORY_ORDERS = {
    "sex": [
        "Female",
        "Male",
    ],
    "race_ethnicity": [
        "Mexican American",
        "Other Hispanic",
        "Non-Hispanic White",
        "Non-Hispanic Black",
        "Non-Hispanic Asian",
        "Other or multiracial",
    ],
    "insurance_history": [
        "Continuously insured",
        "Currently insured, past-year gap",
        "Currently uninsured",
    ],
    "pregnancy_status": [
        "Pregnant",
        "Not pregnant",
        "Could not be determined",
        "Not assessed or structurally missing",
    ],
}

INCOME_GROUP_BINS = [-np.inf, 1, 2, 4, np.inf]
INCOME_GROUP_LABELS = [
    "Below 1",
    "1 to below 2",
    "2 to below 4",
    "4 or higher",
]

complete_case["income_poverty_group"] = pd.cut(
    complete_case["income_poverty_ratio"],
    bins=INCOME_GROUP_BINS,
    labels=INCOME_GROUP_LABELS,
    right=False,
    ordered=True,
)

analysis_base["selection_status"] = np.where(
    analysis_base["id"].isin(complete_case["id"]),
    "Included complete case",
    "Excluded from complete-case analysis",
)

analysis_base["selection_status"] = pd.Categorical(
    analysis_base["selection_status"],
    categories=[
        "Included complete case",
        "Excluded from complete-case analysis",
    ],
    ordered=True,
)


## 6. Helper functions

phlebotomy weights are used for descriptive point estimates only. This notebook does not calculate design-based standard errors from strata and PSUs.

In [ ]:
def valid_weight_mask(
    values: pd.Series,
    weights: pd.Series,
) -> pd.Series:
    return (
        values.notna()
        & weights.notna()
        & pd.to_numeric(weights, errors="coerce").gt(0)
    )


def weighted_mean(
    values: pd.Series,
    weights: pd.Series,
) -> float:
    mask = valid_weight_mask(values, weights)

    if mask.sum() == 0:
        return np.nan

    return float(
        np.average(
            pd.to_numeric(values.loc[mask], errors="coerce"),
            weights=pd.to_numeric(weights.loc[mask], errors="coerce"),
        )
    )


def weighted_sd(
    values: pd.Series,
    weights: pd.Series,
) -> float:
    mask = valid_weight_mask(values, weights)

    if mask.sum() == 0:
        return np.nan

    observed_values = pd.to_numeric(
        values.loc[mask],
        errors="coerce",
    ).to_numpy(dtype=float)

    observed_weights = pd.to_numeric(
        weights.loc[mask],
        errors="coerce",
    ).to_numpy(dtype=float)

    mean_value = np.average(
        observed_values,
        weights=observed_weights,
    )

    variance = np.average(
        (observed_values - mean_value) ** 2,
        weights=observed_weights,
    )

    return float(np.sqrt(variance))


def weighted_quantile(
    values: pd.Series,
    weights: pd.Series,
    quantiles,
):
    quantiles = np.atleast_1d(quantiles).astype(float)

    if ((quantiles < 0) | (quantiles > 1)).any():
        raise ValueError("Quantiles must lie between 0 and 1.")

    mask = valid_weight_mask(values, weights)

    if mask.sum() == 0:
        return np.full(len(quantiles), np.nan)

    observed_values = pd.to_numeric(
        values.loc[mask],
        errors="coerce",
    ).to_numpy(dtype=float)

    observed_weights = pd.to_numeric(
        weights.loc[mask],
        errors="coerce",
    ).to_numpy(dtype=float)

    order = np.argsort(observed_values)
    observed_values = observed_values[order]
    observed_weights = observed_weights[order]

    cumulative_weights = (
        np.cumsum(observed_weights) - 0.5 * observed_weights
    )
    cumulative_weights = cumulative_weights / observed_weights.sum()

    return np.interp(
        quantiles,
        cumulative_weights,
        observed_values,
    )


def weighted_binary_prevalence(
    values: pd.Series,
    weights: pd.Series,
) -> float:
    return weighted_mean(values.astype(float), weights)


def weighted_category_share(
    values: pd.Series,
    weights: pd.Series,
    level,
) -> float:
    mask = values.notna() & weights.notna() & weights.gt(0)

    if mask.sum() == 0:
        return np.nan

    numerator = weights.loc[mask & values.eq(level)].sum()
    denominator = weights.loc[mask].sum()

    return float(numerator / denominator)


def wilson_interval(
    positive_n: int,
    total_n: int,
    z: float = 1.96,
):
    if total_n <= 0:
        return np.nan, np.nan

    proportion = positive_n / total_n
    denominator = 1 + (z**2 / total_n)

    centre = (
        proportion
        + z**2 / (2 * total_n)
    ) / denominator

    half_width = (
        z
        * math.sqrt(
            proportion * (1 - proportion) / total_n
            + z**2 / (4 * total_n**2)
        )
        / denominator
    )

    return centre - half_width, centre + half_width


def preferred_levels(
    data: pd.DataFrame,
    variable: str,
):
    observed = [
        value
        for value in data[variable].dropna().astype(str).unique()
    ]

    preferred = CATEGORY_ORDERS.get(variable, [])

    ordered = [
        value for value in preferred if value in observed
    ]

    ordered.extend(
        sorted(value for value in observed if value not in ordered)
    )

    return ordered


def format_mean_sd(mean_value, sd_value, digits=1):
    return f"{mean_value:.{digits}f} ({sd_value:.{digits}f})"


def format_median_iqr(
    median_value,
    q1_value,
    q3_value,
    digits=2,
):
    return (
        f"{median_value:.{digits}f} "
        f"[{q1_value:.{digits}f}, {q3_value:.{digits}f}]"
    )


def format_n_percent(
    count,
    denominator,
    digits=1,
):
    if denominator == 0:
        return "NA"

    percentage = 100 * count / denominator
    return f"{int(count):,} ({percentage:.{digits}f}%)"


## Part A — Continuous-variable inspection

## 7. Range checks and potential extreme values

Tukey fences are used only to flag observations for inspection. They are **not** used as automatic exclusion rules. Age is top-coded at 80 and income-to-poverty ratio is top-coded at 5.

In [ ]:
continuous_variables = [
    "age",
    "bmi",
    "income_poverty_ratio",
]

documented_ranges = {
    "age": (18, 80),
    "bmi": (0, None),
    "income_poverty_ratio": (0, 5),
}

top_code_values = {
    "age": 80,
    "income_poverty_ratio": 5,
}

range_check_rows = []

for variable in continuous_variables:
    values = pd.to_numeric(
        analysis_base[variable],
        errors="coerce",
    )
    observed = values.dropna()

    q1 = observed.quantile(0.25)
    q3 = observed.quantile(0.75)
    iqr = q3 - q1
    lower_fence = q1 - 1.5 * iqr
    upper_fence = q3 + 1.5 * iqr

    expected_min, expected_max = documented_ranges[variable]

    below_expected = (
        int((observed < expected_min).sum())
        if expected_min is not None
        else 0
    )

    above_expected = (
        int((observed > expected_max).sum())
        if expected_max is not None
        else 0
    )

    top_code = top_code_values.get(variable)
    top_code_n = (
        int(observed.eq(top_code).sum())
        if top_code is not None
        else np.nan
    )

    tiny_positive_placeholder_n = int(
        (
            observed.gt(0)
            & observed.lt(1e-50)
        ).sum()
    )

    range_check_rows.append(
        {
            "variable": variable,
            "adult_base_n": len(analysis_base),
            "observed_n": int(observed.size),
            "missing_n": int(values.isna().sum()),
            "minimum": float(observed.min()),
            "maximum": float(observed.max()),
            "q1": float(q1),
            "q3": float(q3),
            "tukey_lower_fence": float(lower_fence),
            "tukey_upper_fence": float(upper_fence),
            "below_documented_min_n": below_expected,
            "above_documented_max_n": above_expected,
            "tukey_flagged_low_n": int(
                (observed < lower_fence).sum()
            ),
            "tukey_flagged_high_n": int(
                (observed > upper_fence).sum()
            ),
            "top_code_value": top_code,
            "top_code_n": top_code_n,
            "tiny_positive_placeholder_n": tiny_positive_placeholder_n,
            "automatic_exclusion": "No",
        }
    )

continuous_range_checks = pd.DataFrame(range_check_rows)

continuous_range_checks.to_csv(
    TABLE_DIR / "continuous_range_checks_adult_base.csv",
    index=False,
)

continuous_range_checks


## 8. Analytic-sample distribution summaries and reporting choice

The reporting choice is based on the variable's scale, top-coding, and observed shape—not only on a single skewness statistic.

In [ ]:
summary_recommendations = {
    "age": (
        "Use mean (SD) in Table 1. Also retain median and IQR because age "
        "is top-coded at 80."
    ),
    "bmi": (
        "Use mean (SD) in Table 1 as planned, while retaining median and "
        "IQR because BMI can be right-skewed."
    ),
    "income_poverty_ratio": (
        "Use median [IQR] as the primary summary because the variable is "
        "bounded, top-coded at 5, and commonly skewed."
    ),
}

distribution_rows = []

for variable in continuous_variables:
    values = pd.to_numeric(
        complete_case[variable],
        errors="coerce",
    )
    observed = values.dropna()
    weights = complete_case.loc[observed.index, "phlebotomy_weight"]

    weighted_q1, weighted_median, weighted_q3 = weighted_quantile(
        observed,
        weights,
        [0.25, 0.50, 0.75],
    )

    distribution_rows.append(
        {
            "variable": variable,
            "analytic_n": int(observed.size),
            "mean": float(observed.mean()),
            "sd": float(observed.std(ddof=1)),
            "minimum": float(observed.min()),
            "q1": float(observed.quantile(0.25)),
            "median": float(observed.median()),
            "q3": float(observed.quantile(0.75)),
            "maximum": float(observed.max()),
            "skewness": float(observed.skew()),
            "phlebotomy_weighted_mean": weighted_mean(
                observed,
                weights,
            ),
            "phlebotomy_weighted_sd": weighted_sd(
                observed,
                weights,
            ),
            "phlebotomy_weighted_q1": float(weighted_q1),
            "phlebotomy_weighted_median": float(weighted_median),
            "phlebotomy_weighted_q3": float(weighted_q3),
            "recommended_paper_summary": (
                summary_recommendations[variable]
            ),
        }
    )

continuous_distribution_summary = pd.DataFrame(
    distribution_rows
)

continuous_distribution_summary.to_csv(
    TABLE_DIR / "continuous_distribution_summary.csv",
    index=False,
)

continuous_distribution_summary


## 9. Distribution plots

These figures are primarily diagnostic or appendix material. The original observed values—including flagged extremes—remain in the analysis.

In [ ]:
def save_histogram(
    data: pd.DataFrame,
    variable: str,
    bins,
    x_label: str,
    title: str,
    filename: str,
):
    values = pd.to_numeric(
        data[variable],
        errors="coerce",
    ).dropna()

    figure, axis = plt.subplots(figsize=(8, 5))

    axis.hist(
        values,
        bins=bins,
        edgecolor="black",
    )

    axis.axvline(
        values.mean(),
        linestyle="--",
        linewidth=1.5,
        label=f"Mean = {values.mean():.2f}",
    )

    axis.axvline(
        values.median(),
        linestyle=":",
        linewidth=1.5,
        label=f"Median = {values.median():.2f}",
    )

    axis.set_xlabel(x_label)
    axis.set_ylabel("Number of participants")
    axis.set_title(title)
    axis.legend()
    figure.tight_layout()

    output_file = FIGURE_DIR / filename
    figure.savefig(
        output_file,
        dpi=300,
        bbox_inches="tight",
    )

    plt.show()
    plt.close(figure)

    print("Saved:", output_file)


save_histogram(
    complete_case,
    variable="age",
    bins=np.arange(18, 82, 2),
    x_label="Age in years",
    title="Age distribution in the complete-case analytic sample",
    filename="continuous_age_distribution.png",
)

save_histogram(
    complete_case,
    variable="bmi",
    bins=30,
    x_label="Body mass index",
    title="BMI distribution in the complete-case analytic sample",
    filename="continuous_bmi_distribution.png",
)

save_histogram(
    complete_case,
    variable="income_poverty_ratio",
    bins=np.linspace(0, 5, 26),
    x_label="Income-to-poverty ratio",
    title=(
        "Income-to-poverty ratio distribution in the "
        "complete-case analytic sample"
    ),
    filename="continuous_income_poverty_ratio_distribution.png",
)


## Part B — Included-versus-excluded analysis

The adult analysis base contains confirmed current pregnancies so the clinical
eligibility exclusion can be documented explicitly. The complete-case sample
contains only adults with `primary_sample_eligible == 1`.

In [ ]:
excluded_adults = analysis_base.loc[
    analysis_base["selection_status"]
    == "Excluded from complete-case analysis"
].copy()

exclusion_reason_masks = {
    "Confirmed current pregnancy": (
        analysis_base["confirmed_current_pregnancy"] == 1
    ),
    "Did not complete the MEC examination": (
        analysis_base["exam_status"] != 2
    ),
    "Missing or invalid phlebotomy weight": (
        analysis_base["phlebotomy_weight"].isna()
        | analysis_base["phlebotomy_weight"].le(0)
    ),
    "Missing prior-diagnosis target": (
        analysis_base["self_reported_prior_diagnosis"].isna()
    ),
    "Missing current HbA1c target": (
        analysis_base["current_hba1c_ge_6_5"].isna()
    ),
    "Missing age": analysis_base["age"].isna(),
    "Missing sex": analysis_base["sex"].isna(),
    "Missing race/ethnicity": (
        analysis_base["race_ethnicity"].isna()
    ),
    "Missing income-to-poverty ratio": (
        analysis_base["income_poverty_ratio"].isna()
    ),
    "Missing BMI": analysis_base["bmi"].isna(),
    "Missing insurance history": (
        analysis_base["insurance_history"].isna()
    ),
    "Missing survey stratum": (
        analysis_base["survey_stratum"].isna()
    ),
    "Missing survey PSU": analysis_base["survey_psu"].isna(),
}

exclusion_reason_rows = []

excluded_mask = (
    analysis_base["selection_status"]
    == "Excluded from complete-case analysis"
)

for reason, mask in exclusion_reason_masks.items():
    all_adult_n = int(mask.sum())
    excluded_n = int((mask & excluded_mask).sum())

    exclusion_reason_rows.append(
        {
            "reason_nonexclusive": reason,
            "all_adults_n": all_adult_n,
            "excluded_adults_n": excluded_n,
            "share_of_excluded_adults": (
                excluded_n / len(excluded_adults)
            ),
        }
    )

complete_case_selection_summary = pd.DataFrame(
    {
        "group": [
            "Adult analysis base",
            "Primary-sample eligible adults",
            "Confirmed current pregnancies",
            "Included complete cases",
            "Excluded adults",
        ],
        "n": [
            len(analysis_base),
            int(analysis_base["primary_sample_eligible"].sum()),
            int(analysis_base["confirmed_current_pregnancy"].sum()),
            len(complete_case),
            len(excluded_adults),
        ],
        "share_of_adult_base": [
            1.0,
            analysis_base["primary_sample_eligible"].mean(),
            analysis_base["confirmed_current_pregnancy"].mean(),
            len(complete_case) / len(analysis_base),
            len(excluded_adults) / len(analysis_base),
        ],
    }
)

exclusion_reasons = (
    pd.DataFrame(exclusion_reason_rows)
    .sort_values(
        "excluded_adults_n",
        ascending=False,
    )
    .reset_index(drop=True)
)

complete_case_selection_summary.to_csv(
    TABLE_DIR / "complete_case_selection_summary.csv",
    index=False,
)

exclusion_reasons.to_csv(
    TABLE_DIR / "complete_case_exclusion_reasons_nonexclusive.csv",
    index=False,
)

display(complete_case_selection_summary)
display(exclusion_reasons)


## 11. Continuous characteristics of included and excluded adults

For BMI and income, the excluded-group summaries describe only participants with observed values. Their missingness is shown separately and must be considered when interpreting the observed-value comparisons.

Absolute standardised mean differences are descriptive effect-size summaries. They are not hypothesis tests.

In [ ]:
selection_continuous_variables = [
    "age",
    "bmi",
    "income_poverty_ratio",
]

selection_status_order = [
    "Included complete case",
    "Excluded from complete-case analysis",
]

included_excluded_continuous_rows = []

for variable in selection_continuous_variables:
    group_statistics = {}

    for status in selection_status_order:
        group_data = analysis_base.loc[
            analysis_base["selection_status"] == status,
            variable,
        ]

        observed = pd.to_numeric(
            group_data,
            errors="coerce",
        ).dropna()

        group_statistics[status] = {
            "total_n": int(group_data.size),
            "observed_n": int(observed.size),
            "missing_n": int(group_data.isna().sum()),
            "mean": float(observed.mean()),
            "sd": float(observed.std(ddof=1)),
            "median": float(observed.median()),
            "q1": float(observed.quantile(0.25)),
            "q3": float(observed.quantile(0.75)),
            "minimum": float(observed.min()),
            "maximum": float(observed.max()),
        }

    included = group_statistics["Included complete case"]
    excluded = group_statistics[
        "Excluded from complete-case analysis"
    ]

    pooled_sd = math.sqrt(
        (included["sd"] ** 2 + excluded["sd"] ** 2) / 2
    )

    standardised_mean_difference = (
        (included["mean"] - excluded["mean"]) / pooled_sd
        if pooled_sd > 0
        else np.nan
    )

    included_excluded_continuous_rows.append(
        {
            "variable": variable,
            "included_total_n": included["total_n"],
            "included_observed_n": included["observed_n"],
            "included_missing_n": included["missing_n"],
            "included_mean": included["mean"],
            "included_sd": included["sd"],
            "included_median": included["median"],
            "included_q1": included["q1"],
            "included_q3": included["q3"],
            "excluded_total_n": excluded["total_n"],
            "excluded_observed_n": excluded["observed_n"],
            "excluded_missing_n": excluded["missing_n"],
            "excluded_mean": excluded["mean"],
            "excluded_sd": excluded["sd"],
            "excluded_median": excluded["median"],
            "excluded_q1": excluded["q1"],
            "excluded_q3": excluded["q3"],
            "mean_difference_included_minus_excluded": (
                included["mean"] - excluded["mean"]
            ),
            "absolute_standardised_mean_difference": abs(
                standardised_mean_difference
            ),
        }
    )

included_excluded_continuous = pd.DataFrame(
    included_excluded_continuous_rows
)

included_excluded_continuous.to_csv(
    TABLE_DIR / "included_excluded_continuous_comparison.csv",
    index=False,
)

included_excluded_continuous


## 12. Categorical characteristics of included and excluded adults

Percentages for observed categories use the number with an observed value as
the denominator and are reported on a 0–100 percentage scale. Missingness is
reported explicitly.

In [ ]:
selection_categorical_variables = [
    "sex",
    "race_ethnicity",
    "insurance_history",
    "pregnancy_status",
]

included_excluded_categorical_rows = []

for variable in selection_categorical_variables:
    levels = preferred_levels(analysis_base, variable)

    for level in levels:
        row = {
            "variable": variable,
            "level": level,
        }

        for status in selection_status_order:
            status_mask = (
                analysis_base["selection_status"] == status
            )

            values = analysis_base.loc[status_mask, variable]
            observed_n = int(values.notna().sum())
            missing_n = int(values.isna().sum())
            level_n = int(values.eq(level).sum())

            prefix = (
                "included"
                if status == "Included complete case"
                else "excluded"
            )

            row[f"{prefix}_total_n"] = int(status_mask.sum())
            row[f"{prefix}_observed_n"] = observed_n
            row[f"{prefix}_missing_n"] = missing_n
            row[f"{prefix}_level_n"] = level_n
            row[f"{prefix}_percent_among_observed"] = (
                100 * level_n / observed_n
                if observed_n > 0
                else np.nan
            )

        row["percentage_point_difference_included_minus_excluded"] = (
            row["included_percent_among_observed"]
            - row["excluded_percent_among_observed"]
        )

        included_excluded_categorical_rows.append(row)

included_excluded_categorical = pd.DataFrame(
    included_excluded_categorical_rows
)

included_excluded_categorical.to_csv(
    TABLE_DIR / "included_excluded_categorical_comparison.csv",
    index=False,
)

included_excluded_categorical


## 13. Missingness by inclusion status

Because the complete-case definition is based partly on missing predictors and targets, missingness differences are expected. This table makes the selection mechanism transparent.

In [ ]:
selection_missingness_variables = [
    "age",
    "sex",
    "race_ethnicity",
    "pregnancy_status",
    "confirmed_current_pregnancy",
    "primary_sample_eligible",
    "income_poverty_ratio",
    "bmi",
    "insurance_history",
    "self_reported_prior_diagnosis",
    "current_hba1c_ge_6_5",
    "phlebotomy_weight",
    "survey_stratum",
    "survey_psu",
]

selection_missingness_rows = []

for variable in selection_missingness_variables:
    for status in selection_status_order:
        status_values = analysis_base.loc[
            analysis_base["selection_status"] == status,
            variable,
        ]

        selection_missingness_rows.append(
            {
                "variable": variable,
                "selection_status": status,
                "group_n": int(status_values.size),
                "missing_n": int(status_values.isna().sum()),
                "missing_share": float(
                    status_values.isna().mean()
                ),
            }
        )

selection_missingness = pd.DataFrame(
    selection_missingness_rows
)

selection_missingness.to_csv(
    TABLE_DIR / "included_excluded_missingness.csv",
    index=False,
)

selection_missingness


### Interpretation rule for complete-case selection

Differences between included and excluded adults indicate that sample
construction may affect the composition of the analytic sample. They do not by
themselves identify the direction or magnitude of bias in later model
estimates.

The pregnancy exclusion is a prespecified clinical eligibility decision rather
than ordinary predictor missingness. Observed BMI and income values among
excluded participants describe only excluded participants with those variables
observed.

## Part C — Paper-ready analytic-sample description

## 14. Table 1: analytic sample

The weighted column contains Phlebotomy-weighted **complete-case point estimates**. It does not contain survey-design standard errors or confidence intervals.

In [ ]:
table1_rows = []

table1_rows.append(
    {
        "section": "Sample",
        "variable": "Analytic sample",
        "level": "",
        "unweighted_summary": f"{len(complete_case):,}",
        "phlebotomy_weighted_complete_case_estimate": "Not applicable",
        "analytic_n": len(complete_case),
        "notes": (
            "Same non-pregnant complete-case participants are used for both targets."
        ),
    }
)

# Continuous variables
age = complete_case["age"].astype(float)
age_weighted_mean = weighted_mean(
    age,
    complete_case["phlebotomy_weight"],
)
age_weighted_sd = weighted_sd(
    age,
    complete_case["phlebotomy_weight"],
)

table1_rows.append(
    {
        "section": "Continuous predictors",
        "variable": "Age, years",
        "level": "Mean (SD)",
        "unweighted_summary": format_mean_sd(
            age.mean(),
            age.std(ddof=1),
            digits=1,
        ),
        "phlebotomy_weighted_complete_case_estimate": (
            format_mean_sd(
                age_weighted_mean,
                age_weighted_sd,
                digits=1,
            )
        ),
        "analytic_n": int(age.notna().sum()),
        "notes": "Age is top-coded at 80 years.",
    }
)

bmi = complete_case["bmi"].astype(float)
bmi_weighted_mean = weighted_mean(
    bmi,
    complete_case["phlebotomy_weight"],
)
bmi_weighted_sd = weighted_sd(
    bmi,
    complete_case["phlebotomy_weight"],
)

table1_rows.append(
    {
        "section": "Continuous predictors",
        "variable": "BMI",
        "level": "Mean (SD)",
        "unweighted_summary": format_mean_sd(
            bmi.mean(),
            bmi.std(ddof=1),
            digits=1,
        ),
        "phlebotomy_weighted_complete_case_estimate": (
            format_mean_sd(
                bmi_weighted_mean,
                bmi_weighted_sd,
                digits=1,
            )
        ),
        "analytic_n": int(bmi.notna().sum()),
        "notes": "Median and IQR are retained in the diagnostic table.",
    }
)

income = complete_case["income_poverty_ratio"].astype(float)
income_weighted_q1, income_weighted_median, income_weighted_q3 = (
    weighted_quantile(
        income,
        complete_case["phlebotomy_weight"],
        [0.25, 0.50, 0.75],
    )
)

table1_rows.append(
    {
        "section": "Continuous predictors",
        "variable": "Income-to-poverty ratio",
        "level": "Median [Q1, Q3]",
        "unweighted_summary": format_median_iqr(
            income.median(),
            income.quantile(0.25),
            income.quantile(0.75),
            digits=2,
        ),
        "phlebotomy_weighted_complete_case_estimate": (
            format_median_iqr(
                income_weighted_median,
                income_weighted_q1,
                income_weighted_q3,
                digits=2,
            )
        ),
        "analytic_n": int(income.notna().sum()),
        "notes": "Income-to-poverty ratio is top-coded at 5.",
    }
)

# Categorical predictors
categorical_table1_variables = [
    ("sex", "Sex"),
    ("race_ethnicity", "Race/ethnicity"),
    ("insurance_history", "Insurance history"),
]

for variable, display_name in categorical_table1_variables:
    denominator = int(complete_case[variable].notna().sum())

    for level in preferred_levels(complete_case, variable):
        level_n = int(complete_case[variable].eq(level).sum())
        weighted_share = weighted_category_share(
            complete_case[variable],
            complete_case["phlebotomy_weight"],
            level,
        )

        table1_rows.append(
            {
                "section": "Categorical predictors",
                "variable": display_name,
                "level": level,
                "unweighted_summary": format_n_percent(
                    level_n,
                    denominator,
                ),
                "phlebotomy_weighted_complete_case_estimate": (
                    f"{100 * weighted_share:.1f}%"
                ),
                "analytic_n": denominator,
                "notes": "",
            }
        )

# Target prevalence
for target in target_columns:
    denominator = int(complete_case[target].notna().sum())
    positive_n = int(complete_case[target].eq(1).sum())
    weighted_prevalence = weighted_binary_prevalence(
        complete_case[target],
        complete_case["phlebotomy_weight"],
    )

    table1_rows.append(
        {
            "section": "Operational targets",
            "variable": TARGET_DISPLAY_NAMES[target],
            "level": "Positive",
            "unweighted_summary": format_n_percent(
                positive_n,
                denominator,
            ),
            "phlebotomy_weighted_complete_case_estimate": (
                f"{100 * weighted_prevalence:.1f}%"
            ),
            "analytic_n": denominator,
            "notes": (
                "Operational target; neither target is assumed to be a "
                "definitive clinical gold standard."
            ),
        }
    )

table1_analytic_sample = pd.DataFrame(table1_rows)

table1_analytic_sample.to_csv(
    TABLE_DIR / "table1_analytic_sample.csv",
    index=False,
)

table1_analytic_sample


## 15. Reconcile and present target prevalence from Notebook 02

Notebook 02 calculated and saved the prevalence estimates. This cell verifies that the participant counts and estimates still match before creating a paper-ready version.

In [ ]:
recalculated_prevalence_rows = []

for target in target_columns:
    recalculated_prevalence_rows.append(
        {
            "target": target,
            "analytic_sample_n": int(
                complete_case[target].notna().sum()
            ),
            "positive_n": int(complete_case[target].eq(1).sum()),
            "negative_n": int(complete_case[target].eq(0).sum()),
            "unweighted_prevalence": float(
                complete_case[target].mean()
            ),
            "phlebotomy_weighted_complete_case_prevalence": (
                weighted_binary_prevalence(
                    complete_case[target],
                    complete_case["phlebotomy_weight"],
                )
            ),
        }
    )

recalculated_prevalence = pd.DataFrame(
    recalculated_prevalence_rows
)

prevalence_validation = target_prevalence_from_notebook02.merge(
    recalculated_prevalence,
    on="target",
    suffixes=("_notebook02", "_notebook03"),
    validate="one_to_one",
)

count_columns_to_check = [
    "analytic_sample_n",
    "positive_n",
    "negative_n",
]

for column in count_columns_to_check:
    if not prevalence_validation[
        f"{column}_notebook02"
    ].eq(
        prevalence_validation[f"{column}_notebook03"]
    ).all():
        raise ValueError(
            f"Notebook 02 and Notebook 03 disagree on {column}."
        )

estimate_columns_to_check = [
    "unweighted_prevalence",
    "phlebotomy_weighted_complete_case_prevalence",
]

for column in estimate_columns_to_check:
    if not np.allclose(
        prevalence_validation[f"{column}_notebook02"],
        prevalence_validation[f"{column}_notebook03"],
        rtol=1e-10,
        atol=1e-12,
    ):
        raise ValueError(
            f"Notebook 02 and Notebook 03 disagree on {column}."
        )

target_prevalence_paper = recalculated_prevalence.copy()
target_prevalence_paper["target_display_name"] = (
    target_prevalence_paper["target"].map(
        TARGET_DISPLAY_NAMES
    )
)
target_prevalence_paper["unweighted_percent"] = (
    100 * target_prevalence_paper["unweighted_prevalence"]
)
target_prevalence_paper[
    "phlebotomy_weighted_complete_case_percent"
] = (
    100
    * target_prevalence_paper[
        "phlebotomy_weighted_complete_case_prevalence"
    ]
)

target_prevalence_paper = target_prevalence_paper[
    [
        "target",
        "target_display_name",
        "analytic_sample_n",
        "positive_n",
        "negative_n",
        "unweighted_percent",
        "phlebotomy_weighted_complete_case_percent",
    ]
]

target_prevalence_paper.to_csv(
    TABLE_DIR / "target_prevalence_paper.csv",
    index=False,
)

print("Target-prevalence validation passed.")
target_prevalence_paper


## Part D — Label concordance and joint-group interpretation

## 16. The four joint label groups

The discordant groups are not automatically treated as errors. Their meanings depend on treatment, disease history, access to diagnosis, measurement variability, and clinical context.

In [ ]:
label_group_rows = []

for joint_code in JOINT_LABEL_ORDER:
    group_mask = (
        complete_case["joint_label_code"].astype(str)
        == joint_code
    )

    group_n = int(group_mask.sum())
    group_share = group_n / len(complete_case)
    weighted_group_share = (
        complete_case.loc[group_mask, "phlebotomy_weight"].sum()
        / complete_case["phlebotomy_weight"].sum()
    )

    label_group_rows.append(
        {
            "joint_label_code": joint_code,
            "group_display_name": JOINT_LABEL_SHORT[joint_code],
            "n": group_n,
            "unweighted_share": group_share,
            "phlebotomy_weighted_complete_case_share": float(
                weighted_group_share
            ),
            "interpretation": (
                JOINT_LABEL_INTERPRETATION[joint_code]
            ),
        }
    )

label_group_interpretation = pd.DataFrame(
    label_group_rows
)

if label_group_interpretation["n"].sum() != len(complete_case):
    raise ValueError(
        "The joint label-group counts do not sum to the analytic sample."
    )

upstream_group_counts = (
    label_group_counts_from_notebook02["n"]
    .astype(int)
)

current_group_counts = (
    complete_case["label_group"]
    .value_counts()
    .reindex(upstream_group_counts.index)
    .astype(int)
)

if not upstream_group_counts.equals(current_group_counts):
    comparison = pd.concat(
        [
            upstream_group_counts.rename("notebook02_n"),
            current_group_counts.rename("notebook03_n"),
        ],
        axis=1,
    )

    raise ValueError(
        "Notebook 02 and Notebook 03 disagree on label-group counts:\n"
        f"{comparison}"
    )

label_group_interpretation.to_csv(
    TABLE_DIR / "label_group_interpretation.csv",
    index=False,
)

label_group_interpretation


## 17. Paper-ready 2 × 2 concordance table

In [ ]:
label_concordance_counts = pd.crosstab(
    complete_case["self_reported_prior_diagnosis"],
    complete_case["current_hba1c_ge_6_5"],
    rownames=["Prior reported diagnosis"],
    colnames=["Current HbA1c at least 6.5%"],
)

label_concordance_counts = label_concordance_counts.reindex(
    index=[0, 1],
    columns=[0, 1],
    fill_value=0,
)

label_concordance_counts.index = [
    "No prior reported diagnosis",
    "Prior reported diagnosis",
]

label_concordance_counts.columns = [
    "HbA1c below 6.5%",
    "HbA1c at least 6.5%",
]

label_concordance_counts.to_csv(
    TABLE_DIR / "table2_label_concordance.csv"
)

label_concordance_counts


## Part E — Comparison of the four label groups

## 18. Continuous characteristics by joint label group

In [ ]:
group_continuous_rows = []

for joint_code in JOINT_LABEL_ORDER:
    group_data = complete_case.loc[
        complete_case["joint_label_code"].astype(str)
        == joint_code
    ]

    for variable in continuous_variables:
        values = pd.to_numeric(
            group_data[variable],
            errors="coerce",
        )
        observed = values.dropna()
        weights = group_data.loc[
            observed.index,
            "phlebotomy_weight",
        ]

        weighted_q1, weighted_median, weighted_q3 = (
            weighted_quantile(
                observed,
                weights,
                [0.25, 0.50, 0.75],
            )
        )

        group_continuous_rows.append(
            {
                "joint_label_code": joint_code,
                "group_display_name": (
                    JOINT_LABEL_SHORT[joint_code]
                ),
                "variable": variable,
                "group_n": len(group_data),
                "observed_n": int(observed.size),
                "mean": float(observed.mean()),
                "sd": float(observed.std(ddof=1)),
                "median": float(observed.median()),
                "q1": float(observed.quantile(0.25)),
                "q3": float(observed.quantile(0.75)),
                "minimum": float(observed.min()),
                "maximum": float(observed.max()),
                "phlebotomy_weighted_mean": weighted_mean(
                    observed,
                    weights,
                ),
                "phlebotomy_weighted_sd": weighted_sd(
                    observed,
                    weights,
                ),
                "phlebotomy_weighted_median": float(
                    weighted_median
                ),
                "phlebotomy_weighted_q1": float(weighted_q1),
                "phlebotomy_weighted_q3": float(weighted_q3),
            }
        )

label_group_continuous_summary = pd.DataFrame(
    group_continuous_rows
)

label_group_continuous_summary.to_csv(
    TABLE_DIR / "label_group_continuous_summary.csv",
    index=False,
)

label_group_continuous_summary


## 19. Continuous-variable boxplots by joint label group

Visible extreme values are retained. The boxplots are descriptive and should not be interpreted as evidence of causal group differences.

In [ ]:
def save_group_boxplot(
    data: pd.DataFrame,
    variable: str,
    y_label: str,
    title: str,
    filename: str,
):
    values_by_group = []
    labels = []

    for joint_code in JOINT_LABEL_ORDER:
        group_values = pd.to_numeric(
            data.loc[
                data["joint_label_code"].astype(str)
                == joint_code,
                variable,
            ],
            errors="coerce",
        ).dropna()

        values_by_group.append(group_values.to_numpy())
        labels.append(joint_code)

    figure, axis = plt.subplots(figsize=(9, 5))

    axis.boxplot(
        values_by_group,
        showfliers=True,
    )

    axis.set_xticks(
        np.arange(1, len(labels) + 1),
        labels=labels,
    )
    axis.set_xlabel("Joint label group")
    axis.set_ylabel(y_label)
    axis.set_title(title)

    figure.tight_layout()

    output_file = FIGURE_DIR / filename
    figure.savefig(
        output_file,
        dpi=300,
        bbox_inches="tight",
    )

    plt.show()
    plt.close(figure)

    print("Saved:", output_file)


save_group_boxplot(
    complete_case,
    variable="age",
    y_label="Age in years",
    title="Age by joint label group",
    filename="label_group_age_boxplot.png",
)

save_group_boxplot(
    complete_case,
    variable="bmi",
    y_label="Body mass index",
    title="BMI by joint label group",
    filename="label_group_bmi_boxplot.png",
)

save_group_boxplot(
    complete_case,
    variable="income_poverty_ratio",
    y_label="Income-to-poverty ratio",
    title="Income-to-poverty ratio by joint label group",
    filename="label_group_income_poverty_ratio_boxplot.png",
)


## 20. Categorical characteristics by joint label group

Both counts and percentages are retained. Unweighted and Phlebotomy-weighted
percentages are reported on the same 0–100 scale. The weighted percentages are
complete-case point estimates.

In [ ]:
group_categorical_variables = [
    "sex",
    "race_ethnicity",
    "insurance_history",
]

group_categorical_rows = []

for joint_code in JOINT_LABEL_ORDER:
    group_data = complete_case.loc[
        complete_case["joint_label_code"].astype(str)
        == joint_code
    ]

    for variable in group_categorical_variables:
        observed_n = int(group_data[variable].notna().sum())
        missing_n = int(group_data[variable].isna().sum())

        for level in preferred_levels(complete_case, variable):
            level_n = int(group_data[variable].eq(level).sum())

            group_categorical_rows.append(
                {
                    "joint_label_code": joint_code,
                    "group_display_name": (
                        JOINT_LABEL_SHORT[joint_code]
                    ),
                    "variable": variable,
                    "level": level,
                    "group_n": len(group_data),
                    "observed_n": observed_n,
                    "missing_n": missing_n,
                    "level_n": level_n,
                    "percent_among_observed": (
                        100 * level_n / observed_n
                        if observed_n > 0
                        else np.nan
                    ),
                    "phlebotomy_weighted_percent_among_observed": (
                        100
                        * weighted_category_share(
                            group_data[variable],
                            group_data["phlebotomy_weight"],
                            level,
                        )
                    ),
                }
            )

label_group_categorical_summary = pd.DataFrame(
    group_categorical_rows
)

label_group_categorical_summary.to_csv(
    TABLE_DIR / "label_group_categorical_summary.csv",
    index=False,
)

label_group_categorical_summary


## Part F — Focused analysis of the discordant groups

## 21. Treatment variables among participants with prior diagnosis

The main group of interest is `D1_H0`: prior reported diagnosis with current HbA1c below 6.5%. The `D1_H1` group is shown as a descriptive comparison.

Medication use may make the `D1_H0` pattern clinically plausible, but these cross-sectional data cannot prove that treatment caused the current HbA1c value.

In [ ]:
treatment_variables = [
    "insulin_now",
    "diabetes_pills_now",
    "any_diabetes_medication",
]

diagnosed_joint_codes = [
    "D1_H0",
    "D1_H1",
]

treatment_rows = []

for joint_code in diagnosed_joint_codes:
    group_data = complete_case.loc[
        complete_case["joint_label_code"].astype(str)
        == joint_code
    ]

    for variable in treatment_variables:
        observed_values = group_data[variable].dropna().astype(int)
        invalid_values = set(observed_values.unique()).difference(
            {0, 1}
        )

        if invalid_values:
            raise ValueError(
                f"{variable} contains unexpected values: "
                f"{invalid_values}"
            )

        observed_n = int(observed_values.size)
        missing_n = int(group_data[variable].isna().sum())
        yes_n = int(observed_values.eq(1).sum())
        no_n = int(observed_values.eq(0).sum())

        weighted_yes = weighted_binary_prevalence(
            group_data[variable],
            group_data["phlebotomy_weight"],
        )

        treatment_rows.append(
            {
                "joint_label_code": joint_code,
                "group_display_name": (
                    JOINT_LABEL_SHORT[joint_code]
                ),
                "treatment_variable": variable,
                "group_n": len(group_data),
                "observed_n": observed_n,
                "missing_n": missing_n,
                "yes_n": yes_n,
                "no_n": no_n,
                "yes_percent_among_observed": (
                    100 * yes_n / observed_n
                    if observed_n > 0
                    else np.nan
                ),
                "phlebotomy_weighted_yes_percent_among_observed": (
                    100 * weighted_yes
                    if not np.isnan(weighted_yes)
                    else np.nan
                ),
            }
        )

diagnosed_group_treatment_summary = pd.DataFrame(
    treatment_rows
)

diagnosed_group_treatment_summary.to_csv(
    TABLE_DIR / "diagnosed_group_treatment_summary.csv",
    index=False,
)

diagnosed_group_treatment_summary


## 22. Screening and insurance among participants without prior diagnosis

The main group of interest is `D0_H1`: no reported prior diagnosis with current HbA1c at least 6.5%. The `D0_H0` group is shown as a descriptive comparison.

Differences in screening or insurance are associations only. They do not establish that access factors caused the absence of a reported diagnosis.

In [ ]:
no_diagnosis_joint_codes = [
    "D0_H0",
    "D0_H1",
]

screening_access_rows = []

for joint_code in no_diagnosis_joint_codes:
    group_data = complete_case.loc[
        complete_case["joint_label_code"].astype(str)
        == joint_code
    ]

    # Blood-test history
    screening_values = group_data["blood_test_past_3y"]
    screening_observed = screening_values.dropna().astype(int)

    invalid_screening_values = set(
        screening_observed.unique()
    ).difference({0, 1})

    if invalid_screening_values:
        raise ValueError(
            "blood_test_past_3y contains unexpected values: "
            f"{invalid_screening_values}"
        )

    for level_value, level_label in [
        (0, "No"),
        (1, "Yes"),
    ]:
        level_n = int(screening_observed.eq(level_value).sum())
        observed_n = int(screening_observed.size)

        screening_access_rows.append(
            {
                "joint_label_code": joint_code,
                "group_display_name": (
                    JOINT_LABEL_SHORT[joint_code]
                ),
                "variable": "blood_test_past_3y",
                "level": level_label,
                "group_n": len(group_data),
                "observed_n": observed_n,
                "missing_n": int(screening_values.isna().sum()),
                "level_n": level_n,
                "percent_among_observed": (
                    100 * level_n / observed_n
                    if observed_n > 0
                    else np.nan
                ),
                "phlebotomy_weighted_percent_among_observed": (
                    100
                    * weighted_category_share(
                        screening_values,
                        group_data["phlebotomy_weight"],
                        level_value,
                    )
                ),
            }
        )

    # Insurance history
    insurance_observed_n = int(
        group_data["insurance_history"].notna().sum()
    )

    for level in preferred_levels(
        complete_case,
        "insurance_history",
    ):
        level_n = int(
            group_data["insurance_history"].eq(level).sum()
        )

        screening_access_rows.append(
            {
                "joint_label_code": joint_code,
                "group_display_name": (
                    JOINT_LABEL_SHORT[joint_code]
                ),
                "variable": "insurance_history",
                "level": level,
                "group_n": len(group_data),
                "observed_n": insurance_observed_n,
                "missing_n": int(
                    group_data["insurance_history"].isna().sum()
                ),
                "level_n": level_n,
                "percent_among_observed": (
                    100 * level_n / insurance_observed_n
                    if insurance_observed_n > 0
                    else np.nan
                ),
                "phlebotomy_weighted_percent_among_observed": (
                    100
                    * weighted_category_share(
                        group_data["insurance_history"],
                        group_data["phlebotomy_weight"],
                        level,
                    )
                ),
            }
        )

no_diagnosis_screening_access_summary = pd.DataFrame(
    screening_access_rows
)

no_diagnosis_screening_access_summary.to_csv(
    TABLE_DIR / "no_diagnosis_screening_access_summary.csv",
    index=False,
)

no_diagnosis_screening_access_summary


## Part G — Stability and descriptive uncertainty

## 23. Target-positive counts and Wilson intervals within subgroups

The unweighted Wilson intervals describe binomial sampling uncertainty within the analytic sample. They are **not** design-based NHANES confidence intervals.

A cell is flagged when either the positive or negative target count is below 20. Such results should be interpreted cautiously in later fairness analyses.

In [ ]:
subgroup_variables = [
    "sex",
    "race_ethnicity",
    "insurance_history",
    "income_poverty_group",
]

subgroup_target_rows = []

for subgroup_variable in subgroup_variables:
    subgroup_series = complete_case[subgroup_variable]

    if subgroup_variable == "income_poverty_group":
        levels = [
            level
            for level in INCOME_GROUP_LABELS
            if subgroup_series.astype(str).eq(level).any()
        ]
    else:
        levels = preferred_levels(
            complete_case,
            subgroup_variable,
        )

    for level in levels:
        level_mask = subgroup_series.astype(str).eq(str(level))
        level_data = complete_case.loc[level_mask]

        for target in target_columns:
            total_n = int(level_data[target].notna().sum())
            positive_n = int(level_data[target].eq(1).sum())
            negative_n = int(level_data[target].eq(0).sum())

            lower, upper = wilson_interval(
                positive_n,
                total_n,
            )

            weighted_prevalence = weighted_binary_prevalence(
                level_data[target],
                level_data["phlebotomy_weight"],
            )

            if min(positive_n, negative_n) < 20:
                stability_flag = (
                    "Caution: positive or negative count below 20"
                )
            elif min(positive_n, negative_n) < 50:
                stability_flag = (
                    "Moderate cell size: smallest outcome count 20–49"
                )
            else:
                stability_flag = (
                    "At least 50 positive and 50 negative observations"
                )

            subgroup_target_rows.append(
                {
                    "subgroup_variable": subgroup_variable,
                    "subgroup_level": str(level),
                    "target": target,
                    "target_display_name": (
                        TARGET_DISPLAY_NAMES[target]
                    ),
                    "subgroup_n": len(level_data),
                    "positive_n": positive_n,
                    "negative_n": negative_n,
                    "unweighted_prevalence": (
                        positive_n / total_n
                        if total_n > 0
                        else np.nan
                    ),
                    "wilson_95_ci_lower": lower,
                    "wilson_95_ci_upper": upper,
                    "phlebotomy_weighted_complete_case_prevalence": (
                        weighted_prevalence
                    ),
                    "stability_flag": stability_flag,
                }
            )

subgroup_target_stability = pd.DataFrame(
    subgroup_target_rows
)

subgroup_target_stability.to_csv(
    TABLE_DIR / "subgroup_target_prevalence_stability.csv",
    index=False,
)

subgroup_target_stability


## 24. Joint-label counts within subgroups

This table is especially important for the two discordant groups. Categories are not combined automatically. Small cells remain visible and are flagged.

In [ ]:
joint_label_subgroup_rows = []

for subgroup_variable in subgroup_variables:
    subgroup_series = complete_case[subgroup_variable]

    if subgroup_variable == "income_poverty_group":
        levels = [
            level
            for level in INCOME_GROUP_LABELS
            if subgroup_series.astype(str).eq(level).any()
        ]
    else:
        levels = preferred_levels(
            complete_case,
            subgroup_variable,
        )

    for level in levels:
        level_mask = subgroup_series.astype(str).eq(str(level))
        level_data = complete_case.loc[level_mask]
        level_n = len(level_data)

        for joint_code in JOINT_LABEL_ORDER:
            joint_mask = (
                level_data["joint_label_code"].astype(str)
                == joint_code
            )
            cell_n = int(joint_mask.sum())

            weighted_share = (
                level_data.loc[
                    joint_mask,
                    "phlebotomy_weight",
                ].sum()
                / level_data["phlebotomy_weight"].sum()
                if level_n > 0
                else np.nan
            )

            if cell_n < 20:
                stability_flag = "Very small cell: fewer than 20"
            elif cell_n < 50:
                stability_flag = "Small cell: 20–49"
            else:
                stability_flag = "At least 50 observations"

            joint_label_subgroup_rows.append(
                {
                    "subgroup_variable": subgroup_variable,
                    "subgroup_level": str(level),
                    "joint_label_code": joint_code,
                    "group_display_name": (
                        JOINT_LABEL_SHORT[joint_code]
                    ),
                    "subgroup_n": level_n,
                    "cell_n": cell_n,
                    "cell_percent_within_subgroup": (
                        100 * cell_n / level_n
                        if level_n > 0
                        else np.nan
                    ),
                    "phlebotomy_weighted_percent_within_subgroup": (
                        100 * weighted_share
                    ),
                    "stability_flag": stability_flag,
                }
            )

joint_label_counts_by_subgroup = pd.DataFrame(
    joint_label_subgroup_rows
)

joint_label_counts_by_subgroup.to_csv(
    TABLE_DIR / "joint_label_counts_by_subgroup.csv",
    index=False,
)

joint_label_counts_by_subgroup


## Part H — Descriptive sensitivity summaries

## 25. Weighted-versus-unweighted comparison

The differences below show how much selected descriptive point estimates change after applying phlebotomy weights within the complete-case sample. They do not correct for complete-case selection and do not include survey-design uncertainty.

In [ ]:
weighted_unweighted_rows = []

# Continuous summaries
weighted_unweighted_rows.extend(
    [
        {
            "measure": "Age mean",
            "scale": "Years",
            "unweighted_estimate": float(age.mean()),
            "phlebotomy_weighted_complete_case_estimate": (
                age_weighted_mean
            ),
        },
        {
            "measure": "BMI mean",
            "scale": "BMI units",
            "unweighted_estimate": float(bmi.mean()),
            "phlebotomy_weighted_complete_case_estimate": (
                bmi_weighted_mean
            ),
        },
        {
            "measure": "Income-to-poverty ratio median",
            "scale": "Ratio units",
            "unweighted_estimate": float(income.median()),
            "phlebotomy_weighted_complete_case_estimate": float(
                income_weighted_median
            ),
        },
    ]
)

# Target prevalence, expressed in percentage points
for target in target_columns:
    weighted_unweighted_rows.append(
        {
            "measure": TARGET_DISPLAY_NAMES[target],
            "scale": "Percentage points",
            "unweighted_estimate": (
                100 * complete_case[target].mean()
            ),
            "phlebotomy_weighted_complete_case_estimate": (
                100
                * weighted_binary_prevalence(
                    complete_case[target],
                    complete_case["phlebotomy_weight"],
                )
            ),
        }
    )

# Joint-label shares, expressed in percentage points
for joint_code in JOINT_LABEL_ORDER:
    joint_mask = (
        complete_case["joint_label_code"].astype(str)
        == joint_code
    )

    weighted_unweighted_rows.append(
        {
            "measure": (
                f"Joint group {joint_code}: "
                f"{JOINT_LABEL_SHORT[joint_code]}"
            ),
            "scale": "Percentage points",
            "unweighted_estimate": (
                100 * joint_mask.mean()
            ),
            "phlebotomy_weighted_complete_case_estimate": (
                100
                * complete_case.loc[
                    joint_mask,
                    "phlebotomy_weight",
                ].sum()
                / complete_case["phlebotomy_weight"].sum()
            ),
        }
    )

weighted_unweighted_comparison = pd.DataFrame(
    weighted_unweighted_rows
)

weighted_unweighted_comparison[
    "weighted_minus_unweighted"
] = (
    weighted_unweighted_comparison[
        "phlebotomy_weighted_complete_case_estimate"
    ]
    - weighted_unweighted_comparison[
        "unweighted_estimate"
    ]
)

weighted_unweighted_comparison.to_csv(
    TABLE_DIR / "weighted_unweighted_descriptive_comparison.csv",
    index=False,
)

weighted_unweighted_comparison


## 26. Interpretation checklist for the descriptive results

When writing the paper:

- describe the complete-case retention rate and any visible included-versus-excluded differences;
- report both counts and percentages for all four joint label groups;
- call `D0_H1` “no prior diagnosis with current HbA1c at least 6.5%,” not definitively “undiagnosed diabetes”;
- call `D1_H0` “prior diagnosis with current HbA1c below 6.5%,” recognising that treatment or control may explain the pattern;
- treat treatment, screening, income, and insurance comparisons as descriptive associations;
- avoid claiming that an access variable caused non-diagnosis;
- explicitly flag small subgroup cells;
- describe weighted results as Phlebotomy-weighted complete-case point estimates unless a later survey-design analysis calculates appropriate standard errors.

## Part I — Save metadata and final checkpoint

In [ ]:
descriptive_output_files = {
    "continuous_range_checks": (
        TABLE_DIR / "continuous_range_checks_adult_base.csv"
    ),
    "continuous_distribution_summary": (
        TABLE_DIR / "continuous_distribution_summary.csv"
    ),
    "complete_case_selection_summary": (
        TABLE_DIR / "complete_case_selection_summary.csv"
    ),
    "complete_case_exclusion_reasons": (
        TABLE_DIR
        / "complete_case_exclusion_reasons_nonexclusive.csv"
    ),
    "included_excluded_continuous": (
        TABLE_DIR / "included_excluded_continuous_comparison.csv"
    ),
    "included_excluded_categorical": (
        TABLE_DIR / "included_excluded_categorical_comparison.csv"
    ),
    "included_excluded_missingness": (
        TABLE_DIR / "included_excluded_missingness.csv"
    ),
    "table1_analytic_sample": (
        TABLE_DIR / "table1_analytic_sample.csv"
    ),
    "target_prevalence_paper": (
        TABLE_DIR / "target_prevalence_paper.csv"
    ),
    "table2_label_concordance": (
        TABLE_DIR / "table2_label_concordance.csv"
    ),
    "label_group_interpretation": (
        TABLE_DIR / "label_group_interpretation.csv"
    ),
    "label_group_continuous_summary": (
        TABLE_DIR / "label_group_continuous_summary.csv"
    ),
    "label_group_categorical_summary": (
        TABLE_DIR / "label_group_categorical_summary.csv"
    ),
    "diagnosed_group_treatment_summary": (
        TABLE_DIR / "diagnosed_group_treatment_summary.csv"
    ),
    "no_diagnosis_screening_access_summary": (
        TABLE_DIR
        / "no_diagnosis_screening_access_summary.csv"
    ),
    "subgroup_target_prevalence_stability": (
        TABLE_DIR / "subgroup_target_prevalence_stability.csv"
    ),
    "joint_label_counts_by_subgroup": (
        TABLE_DIR / "joint_label_counts_by_subgroup.csv"
    ),
    "weighted_unweighted_comparison": (
        TABLE_DIR
        / "weighted_unweighted_descriptive_comparison.csv"
    ),
}

missing_outputs = [
    str(path)
    for path in descriptive_output_files.values()
    if not path.exists()
]

if missing_outputs:
    raise FileNotFoundError(
        "Some expected descriptive output tables were not saved:\n"
        + "\n".join(missing_outputs)
    )

descriptive_metadata = {
    "adult_analysis_base_n": int(len(analysis_base)),
    "complete_case_n": int(len(complete_case)),
    "excluded_adults_n": int(len(excluded_adults)),
    "confirmed_current_pregnancy_excluded_n": int(
        analysis_base["confirmed_current_pregnancy"].sum()
    ),
    "primary_sample_eligible_adults_n": int(
        analysis_base["primary_sample_eligible"].sum()
    ),
    "complete_case_retained_share": float(
        len(complete_case) / len(analysis_base)
    ),
    "target_positive_counts": {
        target: int(complete_case[target].eq(1).sum())
        for target in target_columns
    },
    "joint_label_counts": {
        joint_code: int(
            complete_case["joint_label_code"]
            .astype(str)
            .eq(joint_code)
            .sum()
        )
        for joint_code in JOINT_LABEL_ORDER
    },
    "income_poverty_group_bins": [
        "-infinity",
        1,
        2,
        4,
        "infinity",
    ],
    "income_poverty_group_labels": INCOME_GROUP_LABELS,
    "weighted_estimate_scope": (
        "Phlebotomy-weighted complete-case point estimates; no "
        "design-based standard errors in this notebook"
    ),
    "automatic_outlier_removal": False,
    "output_tables": {
        name: str(path)
        for name, path in descriptive_output_files.items()
    },
}

descriptive_metadata_path = (
    PROCESSED_DIR / "descriptive_analysis_metadata.json"
)

with descriptive_metadata_path.open(
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        descriptive_metadata,
        file,
        indent=2,
    )

final_checkpoint = {
    "adult_analysis_base_n": len(analysis_base),
    "complete_case_n": len(complete_case),
    "excluded_adults_n": len(excluded_adults),
    "confirmed_current_pregnancy_in_complete_case_n": int(
        complete_case["confirmed_current_pregnancy"].sum()
    ),
    "all_complete_cases_primary_sample_eligible": bool(
        complete_case["primary_sample_eligible"].eq(1).all()
    ),
    "duplicate_complete_case_ids": int(
        complete_case["id"].duplicated().sum()
    ),
    "missing_required_complete_case_values": int(
        complete_case[required_complete_values]
        .isna()
        .sum()
        .sum()
    ),
    "joint_label_count_sum": int(
        label_group_interpretation["n"].sum()
    ),
    "all_expected_tables_saved": len(missing_outputs) == 0,
    "descriptive_metadata_saved": (
        descriptive_metadata_path.exists()
    ),
}

print("Saved descriptive metadata to:")
print(descriptive_metadata_path)
print()
print("Final checkpoint:")
final_checkpoint


## Completion criteria

- Descriptive counts reconcile with upstream sample and target metadata.
- Phlebotomy-weighted values are clearly identified as descriptive complete-case point estimates.
- All expected descriptive files and metadata are saved.